# 📄 arXiv 論文爬蟲 — 從零開始完整教學
### 領域：電腦視覺 / 圖形辨識（Computer Vision & Image Recognition）

---

## 📌 這份 Notebook 的學習路徑

```
階段一：認識 arXiv API 網址長什麼樣
    ↓
階段二：用 Python 發送請求，看看伺服器回傳什麼
    ↓
階段三：了解回傳的 XML 格式
    ↓
階段四：解析 XML，取出我們要的欄位
    ↓
階段五：整理成函式，方便重複使用
    ↓
階段六：執行爬蟲，搜尋 CV 論文，儲存結果
```

---
# 階段一：認識 arXiv API 網址

在寫任何程式之前，先用**瀏覽器**直接打開下面這個網址，看看 API 回傳什麼：

👉 http://export.arxiv.org/api/query?search_query=cat:cs.CV&max_results=2

你會看到一大串 XML 文字，那就是 arXiv 給我們的論文資料。

---

### 網址的組成結構

```
http://export.arxiv.org/api/query
        ↑ 這是 API 的基本網址（Base URL）

?search_query=cat:cs.CV&max_results=2
 ↑ 問號後面是參數，用 & 分隔多個參數
```

| 參數 | 意思 | 範例值 |
|------|------|--------|
| `search_query` | 搜尋什麼 | `cat:cs.CV`（電腦視覺分類） |
| `max_results` | 要幾筆資料 | `10` |
| `start` | 從第幾筆開始 | `0`（第一頁） |
| `sortBy` | 排序方式 | `submittedDate` |
| `sortOrder` | 升冪或降冪 | `descending`（最新優先） |

---
# 階段二：用 Python 發送請求

In [ ]:
# 先確認套件都有安裝
# 在 Jupyter 裡用 ! 開頭可以執行終端機指令
%pip install requests pandas

In [1]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time

print("✅ 所有套件載入成功")

✅ 所有套件載入成功


In [3]:
# ▶ 先用最簡單的方式發送一次請求，只要 2 筆資料
# 目的：確認網路連線正常，API 有回應

url = "http://export.arxiv.org/api/query?search_query=cat:cs.CV&max_results=2"

res = requests.get(url)

print("HTTP 狀態碼：", res.status_code)
# 200 = 成功，404 = 找不到，500 = 伺服器錯誤

HTTP 狀態碼： 200


In [10]:
# ▶ 看看伺服器實際回傳了什麼內容（原始文字）
# 先只印出前 500 個字元，避免畫面太長

raw_text = res.text
print(raw_text[:2000])

<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/WFquG1CEVlfg5AL6NF3pd5hfYDo</id>
  <title>arXiv Query: search_query=cat:cs.CV&amp;id_list=&amp;start=0&amp;max_results=2</title>
  <updated>2026-05-13T11:48:12Z</updated>
  <link href="https://arxiv.org/api/query?search_query=cat:cs.CV&amp;start=0&amp;max_results=2&amp;id_list=" type="application/atom+xml"/>
  <opensearch:itemsPerPage>2</opensearch:itemsPerPage>
  <opensearch:totalResults>191875</opensearch:totalResults>
  <opensearch:startIndex>0</opensearch:startIndex>
  <entry>
    <id>http://arxiv.org/abs/2012.11486v1</id>
    <title>Leaf Segmentation and Counting with Deep Learning: on Model Certainty, Test-Time Augmentation, Trade-Offs</title>
    <updated>2020-12-21T17:00:05Z</updated>
    <link href="https://arxiv.org/abs/2012.11486v1" rel="alternate" type="text/html"/>
 

In [9]:
# ▶ 印出完整的原始 XML（這次全部印出）
# 仔細看看 XML 的結構：哪裡是標題？哪裡是摘要？哪裡是作者？

print(raw_text)

<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/WFquG1CEVlfg5AL6NF3pd5hfYDo</id>
  <title>arXiv Query: search_query=cat:cs.CV&amp;id_list=&amp;start=0&amp;max_results=2</title>
  <updated>2026-05-13T11:48:12Z</updated>
  <link href="https://arxiv.org/api/query?search_query=cat:cs.CV&amp;start=0&amp;max_results=2&amp;id_list=" type="application/atom+xml"/>
  <opensearch:itemsPerPage>2</opensearch:itemsPerPage>
  <opensearch:totalResults>191875</opensearch:totalResults>
  <opensearch:startIndex>0</opensearch:startIndex>
  <entry>
    <id>http://arxiv.org/abs/2012.11486v1</id>
    <title>Leaf Segmentation and Counting with Deep Learning: on Model Certainty, Test-Time Augmentation, Trade-Offs</title>
    <updated>2020-12-21T17:00:05Z</updated>
    <link href="https://arxiv.org/abs/2012.11486v1" rel="alternate" type="text/html"/>
 

---
## 觀察 XML 的結構

執行上一格後，你會看到類似這樣的 XML：

```xml
<?xml version="1.0" encoding="UTF-8"?>
<feed xmlns="http://www.w3.org/2005/Atom">

  <!-- 這是整份文件的基本資訊 -->
  <title>ArXiv Query: ...</title>
  <id>http://arxiv.org/api/...</id>

  <!-- 每篇論文是一個 <entry> 區塊 -->
  <entry>
    <title>論文標題</title>
    <summary>論文摘要...</summary>
    <published>2024-01-15T00:00:00Z</published>
    <author><name>作者姓名</name></author>
    <id>https://arxiv.org/abs/2401.12345</id>
  </entry>

  <!-- 第二篇論文 -->
  <entry>...</entry>

</feed>
```

所以我們的任務就是：**把每個 `<entry>` 裡面我們要的欄位取出來**

---
# 階段三：認識 XML 的命名空間（Namespace）

你有沒有注意到 XML 第一行有這個？

```xml
<feed xmlns="http://www.w3.org/2005/Atom">
```

`xmlns` 是 **XML Namespace（命名空間）** 的宣告。

意思是：這份 XML 裡所有標籤，都屬於 `http://www.w3.org/2005/Atom` 這個規範。

所以 Python 內部看到的標籤其實長這樣：

```
你看到的       Python 內部實際的完整名稱
<title>   →   {http://www.w3.org/2005/Atom}title
<summary> →   {http://www.w3.org/2005/Atom}summary
<entry>   →   {http://www.w3.org/2005/Atom}entry
```

In [11]:
# ▶ 解析 XML，看看 root 節點長什麼樣

root = ET.fromstring(res.content)
# response.content 是 bytes（原始二進位）
# ET.fromstring() 把它解析成樹狀結構

print("root 的標籤名稱：", root.tag)
# 你會看到完整的命名空間被包在 {} 裡面

root 的標籤名稱： {http://www.w3.org/2005/Atom}feed


In [ ]:
# ▶ 看看 root 底下有哪些子標籤

for child in root:
    print(child.tag)

In [ ]:
# ▶ 示範：不寫 namespace 會發生什麼事？

# ❌ 錯誤做法：找不到任何 entry
entries_wrong = root.findall("entry")
print("不寫 namespace，找到幾個 entry：", len(entries_wrong))
# 結果是 0，完全找不到！

In [ ]:
# ▶ 正確做法一：寫完整的命名空間（很長，不好閱讀）

entries_correct = root.findall("{http://www.w3.org/2005/Atom}entry")
print("寫完整 namespace，找到幾個 entry：", len(entries_correct))

In [ ]:
# ▶ 正確做法二：用字典幫命名空間取別名「atom」（我們採用這種）

namespace = {"atom": "http://www.w3.org/2005/Atom"}
# 之後寫 "atom:entry" 就等於 "{http://www.w3.org/2005/Atom}entry"

entries = root.findall("atom:entry", namespace)
print("用別名 namespace，找到幾個 entry：", len(entries))

---
# 階段四：逐步從 XML 取出每個欄位

In [ ]:
# ▶ 先只看第一篇論文（entries[0]）
# 確認各欄位取出來的值是否正確

first_entry = entries[0]
print("第一篇論文的 entry 標籤：", first_entry.tag)

In [ ]:
# ▶ 取出標題

title = first_entry.find("atom:title", namespace).text
print("原始標題（未處理）：", repr(title))
# repr() 可以看到隱藏的換行符號 \n 和空白

title_clean = title.strip()  # 去除前後空白和換行
print("清理後標題：", title_clean)

In [ ]:
# ▶ 取出摘要（Abstract）

summary = first_entry.find("atom:summary", namespace).text.strip()
print("摘要（前 200 字）：")
print(summary[:200], "...")

In [ ]:
# ▶ 取出投稿日期

published_raw = first_entry.find("atom:published", namespace).text
print("原始日期：", published_raw)
# 格式是 2024-01-15T00:00:00Z

published_clean = published_raw[:10]  # 只取前 10 字元
print("清理後日期：", published_clean)

In [ ]:
# ▶ 取出作者（可能有多位）

author_tags = first_entry.findall("atom:author", namespace)
print("作者數量：", len(author_tags))

# 用 list comprehension 取出每位作者的名字
authors = [a.find("atom:name", namespace).text for a in author_tags]
print("作者列表：", authors)

In [ ]:
# ▶ 取出論文連結

link = first_entry.find("atom:id", namespace).text
print("論文連結：", link)

In [ ]:
# ▶ 把第一篇論文的所有欄位整合成一個字典

paper = {
    "title"    : first_entry.find("atom:title", namespace).text.strip(),
    "summary"  : first_entry.find("atom:summary", namespace).text.strip(),
    "published": first_entry.find("atom:published", namespace).text[:10],
    "authors"  : [a.find("atom:name", namespace).text
                  for a in first_entry.findall("atom:author", namespace)],
    "link"     : first_entry.find("atom:id", namespace).text
}

# 印出這個字典
for key, value in paper.items():
    print(f"\n【{key}】")
    print(value)

In [ ]:
# ▶ 把所有 entry 都處理（用 for 迴圈）

papers = []  # 空串列，用來存所有論文

for entry in entries:
    paper = {
        "title"    : entry.find("atom:title", namespace).text.strip(),
        "summary"  : entry.find("atom:summary", namespace).text.strip(),
        "published": entry.find("atom:published", namespace).text[:10],
        "authors"  : [a.find("atom:name", namespace).text
                      for a in entry.findall("atom:author", namespace)],
        "link"     : entry.find("atom:id", namespace).text
    }
    papers.append(paper)

print(f"✅ 共取出 {len(papers)} 篇論文")
print("第一篇標題：", papers[0]["title"])
print("第二篇標題：", papers[1]["title"])

---
# 階段五：把以上步驟整理成一個函式

上面的步驟每次要爬新的關鍵字都要重寫一遍，很麻煩。

所以我們把它**包成一個函式**，之後只要呼叫函式就好。

In [ ]:
def fetch_arxiv_papers(query, max_results=10):
    """
    從 arXiv API 爬取論文資料

    參數：
        query      : 搜尋關鍵字，例如 'cat:cs.CV AND image recognition'
        max_results: 最多回傳幾筆，預設 10

    回傳：
        papers: 論文資料的串列（每篇論文是一個字典）
    """

    # --- 步驟 1：設定 API 網址與參數 ---
    base_url = "http://export.arxiv.org/api/query"
    params = {
        "search_query": query,
        "start"       : 0,
        "max_results" : max_results,
        "sortBy"      : "submittedDate",
        "sortOrder"   : "descending"
    }

    # --- 步驟 2：發送 GET 請求 ---
    response = requests.get(base_url, params=params)

    # --- 步驟 3：解析 XML ---
    root = ET.fromstring(response.content)
    namespace = {"atom": "http://www.w3.org/2005/Atom"}

    # --- 步驟 4：逐篇取出資料 ---
    papers = []
    for entry in root.findall("atom:entry", namespace):
        paper = {
            "title"    : entry.find("atom:title", namespace).text.strip(),
            "summary"  : entry.find("atom:summary", namespace).text.strip(),
            "published": entry.find("atom:published", namespace).text[:10],
            "authors"  : [a.find("atom:name", namespace).text
                          for a in entry.findall("atom:author", namespace)],
            "link"     : entry.find("atom:id", namespace).text
        }
        papers.append(paper)

    return papers

print("✅ 函式定義完成，可以開始使用")

---
# 階段六：執行爬蟲，搜尋電腦視覺論文

In [ ]:
# ▶ 單一關鍵字搜尋

papers = fetch_arxiv_papers(
    query="cat:cs.CV AND (image recognition OR object detection)",
    max_results=10
)

print(f"✅ 共爬取到 {len(papers)} 篇論文")

In [ ]:
# ▶ 轉成 DataFrame，顯示標題和日期

df = pd.DataFrame(papers)
print(df[["title", "published"]].to_string())

In [ ]:
# ▶ 在 Jupyter 裡直接顯示完整表格（會有漂亮的格線）
df

In [ ]:
# ▶ 查看第一篇論文的完整內容

first = papers[0]
print("📌 標題  ：", first["title"])
print("📅 日期  ：", first["published"])
print("👤 作者  ：", ", ".join(first["authors"]))
print("🔗 連結  ：", first["link"])
print("\n📝 摘要：")
print(first["summary"])

In [ ]:
# ▶ 多個關鍵字搜尋（這裡才需要 time.sleep）
# 多次請求之間加 3 秒延遲，避免對伺服器造成負擔

queries = [
    "cat:cs.CV AND image recognition",
    "cat:cs.CV AND object detection",
    "cat:cs.CV AND image segmentation"
]

all_papers = []

for q in queries:
    print(f"🔍 搜尋中：{q}")
    result = fetch_arxiv_papers(query=q, max_results=5)
    all_papers.extend(result)
    time.sleep(3)  # 等 3 秒再發下一次請求

# 去除重複論文（同一篇可能被多個關鍵字搜到）
df_all = pd.DataFrame(all_papers).drop_duplicates(subset="link")

print(f"\n✅ 總共爬取到 {len(df_all)} 篇不重複論文")

In [ ]:
# ▶ 儲存成 CSV
# utf-8-sig：讓 Excel 開啟時中文不會亂碼

df_all.to_csv("cv_papers.csv", index=False, encoding="utf-8-sig")
print(f"✅ 已儲存！共 {len(df_all)} 筆論文 → cv_papers.csv")